### Stacking algorithm - classification example, trying CatBoost also

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

import tensorflow as tf
import keras
from keras import layers

from sklearn.preprocessing import label_binarize


In [2]:
# load credit group date, replace this with your own .csv -data
# this version of data should be READY TO GO to training
# which means => only numeric values in the dataset
df = pd.read_csv("processed_classification_credit_group.csv")
df.head()

,Delay_from_due_date,Num_of_Delayed_Payment,Num_Credit_Inquiries,Credit_Utilization_Ratio,Credit_History_Age,Amount_invested_monthly,Monthly_Balance,Credit_Mix,Age,Annual_Income,...,Outstanding_Debt,Total_EMI_per_month,Payment_of_Min_Amount_No,Payment_of_Min_Amount_Yes,Payment_Behaviour_High_spent_Large_value_payments,Payment_Behaviour_High_spent_Medium_value_payments,Payment_Behaviour_High_spent_Small_value_payments,Payment_Behaviour_Low_spent_Large_value_payments,Payment_Behaviour_Low_spent_Medium_value_payments,Credit_Score
0,3.0,7.0,4.0,26.822620,265.0,80.415295,312.494089,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
1,3.0,7.0,4.0,31.944960,265.0,118.280222,284.629162,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
2,3.0,7.0,4.0,28.609352,267.0,81.699521,331.209863,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
3,5.0,4.0,4.0,31.377862,268.0,199.458074,223.451310,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2
4,6.0,4.0,4.0,24.797347,269.0,41.420153,341.489231,2,23.0,19114.12,...,809.98,49.574949,1,0,0,1,0,0,0,2


In [3]:
target = "Credit_Score"

# CatBoost requires we save our continuous and categorical variables separately into lists
categorical_variables = ["Credit_Mix", 'Payment_of_Min_Amount_No', 'Payment_of_Min_Amount_Yes',
                        'Payment_Behaviour_High_spent_Large_value_payments',
                        'Payment_Behaviour_High_spent_Medium_value_payments',
                        'Payment_Behaviour_High_spent_Small_value_payments',
                        'Payment_Behaviour_Low_spent_Large_value_payments',
                        'Payment_Behaviour_Low_spent_Medium_value_payments']

# the usual X/y -split
X = df.drop(target, axis=1)
y = df[target]

# usual train/test -split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# NOTE! SCALING => some of the algorithms require this
scaler = StandardScaler()

# create separate versions for the scaled data
# because we need both unscaled and scaled versions later
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### Create a stacking algorithm for classification

In [4]:
from sklearn.ensemble import StackingClassifier

# meta model is the "judge" or the optimizer of the stacking algorithm
# => it assigns weights to the models based on their performance
meta_model = LogisticRegression(class_weight="balanced")

# "Random Forest": RandomForestClassifier(n_estimators=100),
# "XGBoost": xgb.XGBClassifier(enable_categorical=True, objective="multi:softprob", num_class=len(np.unique(y_train))),
# "CatBoost-default": cb.CatBoostClassifier(iterations=500, verbose=0),
# "CatBoost-opt": cb.CatBoostClassifier(learning_rate=np.float64(0.14), iterations=np.float64(600.0), depth=7, verbose=0),
# 'LightGBM': lgb.LGBMClassifier(objective='multiclass')

# stacking regression algorithm
stacking_model = StackingClassifier(
    estimators=[('xgb', xgb.XGBClassifier(enable_categorical=True, objective="multi:softprob", num_class=len(np.unique(y_train)))),
                ('randomforest', RandomForestClassifier(n_estimators=100)),
                ('catboost', cb.CatBoostClassifier(iterations=675, learning_rate=0.234285714285714, depth=7, verbose=0)),
                ('lgbm', lgb.LGBMClassifier(objective='multiclass'))],
    final_estimator=meta_model,
    cv=5
)

# other than that, the stacking algorithm works as any other algorithm
stacking_model.fit(X_train, y_train)

# test predictions and metrics
predictions = stacking_model.predict(X_test)

probabilities = stacking_model.predict_proba(X_test)

accuracy = stacking_model.score(X_test, y_test)
print(f"\nAccuracy: {accuracy}")

# multiclass ROC-AUC
roc_auc = roc_auc_score(y_test, probabilities, multi_class="ovr", average="macro")
print(f"ROC-AUC: {roc_auc}")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002213 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2536
[LightGBM] [Info] Number of data points in the train set: 65869, number of used features: 25
[LightGBM] [Info] Start training from score -1.280330
[LightGBM] [Info] Start training from score -0.631577
[LightGBM] [Info] Start training from score -1.659143
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001670 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2534
[LightGBM] [Info] Number of data points in the train set: 52695, number of used features: 25
[LightGBM] [Info] Start training from score -1.280285
[LightGBM] [Info] Start training from score -0.631602
[LightGBM] [Info] Start training from score -1.659139
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001897 sec

In [ ]:
# adding catboost into the mix didn't seem to improve this time

# without catboost, metrics were:
# Accuracy: 0.8823172212776293
# ROC-AUC: 0.9615338220219017

# then again, the stacking algorithm seems to use cross validation
# for the metrics, so these actually might be very good results!